# QDAC sourced NI DAQ CrossTalk Frequency Depdendency Test

## Import Libraries

In [1]:
import time
import json
import pyvisa
import numpy as np
import matplotlib.pyplot as plt
from time import sleep

from qcodes import (
    Parameter,
    Measurement,
    Station,
    load_or_create_experiment,
    initialise_or_create_database_at,
)
from qcodes.instrument.channel import ChannelList

from qstl_instruments.qstl_qdac2 import QSTL_QDac2
from qstl_instruments.qstl_nidaq import QSTL_NIDaq

from pyvisa.constants import StopBits, Parity

rm = pyvisa.ResourceManager()
rm.list_resources()

('USB0::0x0957::0x1780::MY60101437::INSTR',
 'ASRL1::INSTR',
 'ASRL3::INSTR',
 'ASRL5::INSTR',
 'ASRL6::INSTR',
 'ASRL20::INSTR',
 'ASRL21::INSTR',
 'ASRL22::INSTR',
 'ASRL23::INSTR',
 'ASRL24::INSTR',
 'ASRL25::INSTR',
 'ASRL26::INSTR',
 'ASRL27::INSTR')

## Instantiation of Instruments

In [2]:
contacts = {
    "X" : 1,
    "Y" : 2
}

ai_chans = {
    "I1" : "Dev2/ai0",
    "I2" : "Dev2/ai1",
    "I3" : "Dev2/ai2"
}

# Set up database
initialise_or_create_database_at("C:/Users/Measurement6/Nextcloud2/Lab/Data/QSim/2025/20251009_QDAC_NIQAC_synctest/QDAC_IQ_Mod_Bandwidth_Test.db")

qdac2 = QSTL_QDac2(
    name = "qdac2",
    address = "ASRL5::INSTR",
    ramp_rate = 1,
    i_threshold = 2e-9,
    v_limit = 0.5,
    contacts = contacts
)
station = Station(qdac2)

daq = QSTL_NIDaq(
    max_sampling_rate = int(1e6),
    gain = 1e8
)

qdac2.ramp_all_channels_to_zero()
qdac2.get_initial_voltages()

Connected to: QDevil QDAC-II (serial:368, firmware:13-1.57) in 0.09s


{'X': 0.0, 'Y': 0.0}

## Instruments Setup

In [119]:
## setup qdac 2 for the 2D sweep
qdac2.free_all_triggers()
qdac2.ext3.delay_s(0)
qdac2.v_limit = 1.2

device1 = "I1"
device2 = "I2"

X = ["X"]
Y = ["Y"]

freqs = np.linspace(1000, 400000, 50)
acq_time = 2000e-6
times = np.linspace(0, acq_time, int(acq_time * daq.max_sampling_rate / 2))

qdac2.channels[0:23].dc_slew_rate_V_per_s("inf")

exp = load_or_create_experiment("2D sweep", "QDAC+NiDAQ_Xtalk_Freq_Dep_Test")
meas = Measurement(exp=exp, station=station)

I1 = Parameter(name= "I1", label="I1", unit="V")
I2 = Parameter(name= "I2", label="I2", unit="V")
R = Parameter(name = "rho_xy", label = "rho_xy", unit = "AU")
t = Parameter(name = "t",label="t", unit = "s" )
f = Parameter(name = "f",label="f", unit = "Hz" )

meas.register_parameter(t)
meas.register_parameter(f)
meas.register_parameter(I1, setpoints = (t, f))
meas.register_parameter(I2, setpoints = (t, f))
meas.register_parameter(R, setpoints = (t, f))

## Correlation Function

In [110]:
def crosscorr_rho(x: np.array, y: np.array) -> np.array:
    x = np.asarray(x)
    y = np.asarray(y)

    N = x.size

    lags = np.linspace(0, N-1, N,  dtype=int)

    px  = np.concatenate(([0], np.cumsum(x)))
    py  = np.concatenate(([0], np.cumsum(y)))
    px2 = np.concatenate(([0], np.cumsum(np.abs(x)**2)))
    py2 = np.concatenate(([0], np.cumsum(np.abs(y)**2)))

    rho = np.empty_like(lags, dtype=np.float64)
    rxx = np.vdot(x, x)
    ryy = np.vdot(y, y)

    for idx, k in enumerate(lags):
        n = N - k
        x0, x1 = 0, n - 1
        y0, y1 = k, N - 1

        x_seg = x[x0:x1]
        y_seg = y[y0:y1]

        rxy = np.vdot(x[x0:x1], y[y0:y1])  # vdot does conjugate on the first argument

        rho[idx] = rxy/np.sqrt(np.abs(rxx * ryy))
    return rho


## X Measurement

In [ ]:
qdac2.free_all_triggers()
InitialConditions = qdac2.get_initial_voltages()

start_time = time.time()
with meas.run() as datasaver:
    loop_counter = 0
    datasaver.dataset.add_metadata(tag="Contacts", metadata=json.dumps(contacts))
    datasaver.dataset.add_metadata(tag="IC", metadata=json.dumps(InitialConditions))
    datasaver.dataset.add_metadata(
        tag = "Sweep_params",
        metadata = json.dumps(
            {
                "X" : X,
                "Y" : Y,
                "freqs" : list(freqs),
                "acq_time" : acq_time
            }
        )
    )
    for freq in freqs:
        qdac2.ramp_all_channels_to_zero()
        qdac2.free_all_triggers()
        sine = qdac2.ch01.sine_wave(
            period_s = 1/freq,
            span_V = 1,
            offset_V = 0.0
        )
        trig = sine.start_marker()
        qdac2.ext5.width_s(2e-6)
        qdac2.ext5.source_from_trigger(trig)
        # Read NI Daq traces
        result = daq.read_triggered_multi_channels(
            sine,
            [ai_chans[device1], ai_chans[device2]],
            int(acq_time * daq.max_sampling_rate / 2),
            -1,
            +1,
            int(acq_time * daq.max_sampling_rate / 2) + 1
        )
        result_0 = result[0]
        result_1 = result[1]

        datasaver.add_result(
            (t, np.linspace(0, acq_time, len(result_0))),
            (f, [freq] * len(result_0)),
            (R, crosscorr_rho(result_1, result_0)),
            (I1, result_0),
            (I2, result_1),
        )  
            
        loop_counter = loop_counter+1
        print(f'Time elapsed: {np.round(time.time()-start_time, 2)} sec. Loop finished: {loop_counter}/{len(freqs)}.')

end_time = time.time()
print(f'Time elapsed: {np.round(end_time-start_time, 2)} sec.')

qdac2.channels[0:23].dc_slew_rate_V_per_s(1)

qdac2.ramp_all_channels_to_zero()

Starting experimental run with id: 136. 
Time elapsed: 0.06 sec. Loop finished: 1/50.
Time elapsed: 0.09 sec. Loop finished: 2/50.
Time elapsed: 0.12 sec. Loop finished: 3/50.
Time elapsed: 0.15 sec. Loop finished: 4/50.
Time elapsed: 0.18 sec. Loop finished: 5/50.
Time elapsed: 0.21 sec. Loop finished: 6/50.
Time elapsed: 0.25 sec. Loop finished: 7/50.
Time elapsed: 0.29 sec. Loop finished: 8/50.
Time elapsed: 0.32 sec. Loop finished: 9/50.
Time elapsed: 0.36 sec. Loop finished: 10/50.
Time elapsed: 0.39 sec. Loop finished: 11/50.
Time elapsed: 0.42 sec. Loop finished: 12/50.
Time elapsed: 0.45 sec. Loop finished: 13/50.
Time elapsed: 0.48 sec. Loop finished: 14/50.
Time elapsed: 0.51 sec. Loop finished: 15/50.
Time elapsed: 0.54 sec. Loop finished: 16/50.
Time elapsed: 0.58 sec. Loop finished: 17/50.
Time elapsed: 0.61 sec. Loop finished: 18/50.
Time elapsed: 0.64 sec. Loop finished: 19/50.
Time elapsed: 0.67 sec. Loop finished: 20/50.
Time elapsed: 0.7 sec. Loop finished: 21/50.
Tim